In [3]:
import pandas as pd
from IPython.display import display

input_file = "master_sppg_final.csv"
output_file = "master_sppg_final_uppercase.csv"

kolom_uppercase = [
    "Nama SPPG",
    "Provinsi SPPG",
    "Kabupaten/Kota SPPG",
    "Kecamatan SPPG",
    "Kelurahan SPPG",
]

df = pd.read_csv(input_file, sep=";")

kolom_tidak_ditemukan = [kolom for kolom in kolom_uppercase if kolom not in df.columns]
if kolom_tidak_ditemukan:
    raise ValueError(f"Kolom tidak ditemukan: {kolom_tidak_ditemukan}")

for kolom in kolom_uppercase:
    df[kolom] = df[kolom].astype("string").str.upper()

df.to_csv(output_file, index=False, sep=";")

print(f"File berhasil disimpan: {output_file} ({len(df):,} baris)")
display(df.head())

File berhasil disimpan: master_sppg_final_uppercase.csv (29,119 baris)


,No SPPG,Nama SPPG,Provinsi SPPG,Kabupaten/Kota SPPG,Kecamatan SPPG,Kelurahan SPPG,Alamat SPPG,Latitude,Longitude,link
0,1,SPPG BANDUNG PACET MEKARSARI 1,JAWA BARAT,BANDUNG,PACET,MEKARSARI,"KP, JL. TENJONAGARA, KP. TENJONAGARA, MEKARJAY...",-7.098274,107.701779,https://www.google.com/maps/place/SPPG+BANDUNG...
1,2,SPPG BANDUNG PACET MARUYUNG 1,JAWA BARAT,BANDUNG,PACET,MARUYUNG,"SPPG BANDUNG PACET MARUYUNG 1, JL. RAYA PACET,...",-7.095649,107.706490,https://www.google.com/maps/place/SPPG+BANDUNG...
2,3,SPPG PANGAUBAN PACET 05,JAWA BARAT,BANDUNG,PACET,PANGAUBAN,"SPPG PANGAUBAN PACET 05, KP CANDI, RT.03/RW.06...",-7.134224,107.687835,https://www.google.com/maps/place/SPPG+PANGAUB...
3,4,SPPG PACET#008,JAWA BARAT,BANDUNG,CIPARAY,CIKONENG,"SPPG PACET008, CIPEUJEUH, KEC. PACET, KABUPATE...",-7.087087,107.706797,https://www.google.com/maps/place/SPPG+PACET%2...
4,5,SPPG PACET CIKAWAO 2,JAWA BARAT,BANDUNG,PACET,CIKAWAO,"SPPG PACET CIKAWAO 2, RT.04/RW.12, CIKAWAO, KE...",-7.089001,107.741236,https://www.google.com/maps/place/Sppg+pacet+c...


In [4]:
df_rapi = df.copy()

for kolom in df_rapi.select_dtypes(include=["object", "string"]).columns:
    df_rapi[kolom] = (
        df_rapi[kolom]
        .str.replace(r"[:;]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

output_file = "master_sppg_final_rapi.csv"
df_rapi.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"File berhasil disimpan: {output_file}")
display(df_rapi.head())

File berhasil disimpan: master_sppg_final_rapi.csv


,No SPPG,Nama SPPG,Provinsi SPPG,Kabupaten/Kota SPPG,Kecamatan SPPG,Kelurahan SPPG,Alamat SPPG,Latitude,Longitude,link
0,1,SPPG BANDUNG PACET MEKARSARI 1,JAWA BARAT,BANDUNG,PACET,MEKARSARI,"KP, JL. TENJONAGARA, KP. TENJONAGARA, MEKARJAY...",-7.098274,107.701779,https//www.google.com/maps/place/SPPG+BANDUNG+...
1,2,SPPG BANDUNG PACET MARUYUNG 1,JAWA BARAT,BANDUNG,PACET,MARUYUNG,"SPPG BANDUNG PACET MARUYUNG 1, JL. RAYA PACET,...",-7.095649,107.706490,https//www.google.com/maps/place/SPPG+BANDUNG+...
2,3,SPPG PANGAUBAN PACET 05,JAWA BARAT,BANDUNG,PACET,PANGAUBAN,"SPPG PANGAUBAN PACET 05, KP CANDI, RT.03/RW.06...",-7.134224,107.687835,https//www.google.com/maps/place/SPPG+PANGAUBA...
3,4,SPPG PACET#008,JAWA BARAT,BANDUNG,CIPARAY,CIKONENG,"SPPG PACET008, CIPEUJEUH, KEC. PACET, KABUPATE...",-7.087087,107.706797,https//www.google.com/maps/place/SPPG+PACET%23...
4,5,SPPG PACET CIKAWAO 2,JAWA BARAT,BANDUNG,PACET,CIKAWAO,"SPPG PACET CIKAWAO 2, RT.04/RW.12, CIKAWAO, KE...",-7.089001,107.741236,https//www.google.com/maps/place/Sppg+pacet+ci...


In [5]:
import json

records = json.loads(df_rapi.to_json(orient="records", force_ascii=False))

features = []
for row in records:
    latitude = row.pop("Latitude")
    longitude = row.pop("Longitude")

    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [longitude, latitude],
        },
        "properties": row,
    })

geojson = {
    "type": "FeatureCollection",
    "features": features,
}

output_geojson = "master_sppg_final.geojson"
with open(output_geojson, "w", encoding="utf-8") as file:
    json.dump(geojson, file, ensure_ascii=False, indent=2)

print(f"GeoJSON berhasil disimpan: {output_geojson} ({len(features):,} fitur)")

GeoJSON berhasil disimpan: master_sppg_final.geojson (29,119 fitur)
